In [ ]:

from micom import Community
import pandas as pd
import os
import logging
from micom.solution import solve, add_pfba_objective, optimize_with_retry
from micom.util import check_modification, interface_to_str, _format_min_growth, _apply_min_growth
from micom.problems import regularize_l2_norm
from micom.community import cooperative_tradeoff
from collections.abc import Sized
import numpy as np

In [ ]:
model_dir1 = "./final/HvSC1"

model_dir2 = "./final/HvSC2"

media_dir = "./media_creation/created_media"
taxonomy_df = pd.read_csv("./communities/SynComs_taxonomy.csv")


In [ ]:
def create_medium(media_df):
    media_dict = dict(zip(media_df.iloc[:,0], media_df.iloc[:,1]))
    return media_dict

In [ ]:
def cooperative_tradeoff_nocross(community, min_growth, fraction, fluxes, pfba, atol, rtol): #this is the normal cooperative_tradeoff function from MICOM, just without the crossover
    """Find the best tradeoff between community and individual growth."""
    with community as com:
        solver = interface_to_str(community.problem)
        check_modification(community)
        min_growth = _format_min_growth(min_growth, community.taxa)
        _apply_min_growth(community, min_growth)

        com.objective = com.scale * com.variables.community_objective
        min_growth = (
            optimize_with_retry(com, message="could not get community growth rate.")
            / com.scale
        )
        if not isinstance(fraction, Sized):
            fraction = [fraction]
        else:
            fraction = np.sort(fraction)[::-1]

        # Add needed variables etc.
        regularize_l2_norm(com, 0.0)
        results = []
        for fr in fraction:
            com.variables.community_objective.lb = fr * min_growth
            com.variables.community_objective.ub = min_growth
            sol = solve(community, fluxes=fluxes, pfba=pfba, atol=atol, rtol=rtol)
            #if not pfba and sol.status != OPTIMAL:
               # sol = crossover(com, sol, fluxes=fluxes)
            results.append((fr, sol))
        if len(results) == 1:
            return results[0][1]
        return pd.DataFrame.from_records(results, columns=["tradeoff", "solution"])

In [ ]:
def cooperative_tradeoff_custom(community, min_growth, fraction):
    """Find the best tradeoff between community and individual growth."""
    with community as com:
        check_modification(community)
        if not isinstance(fraction, Sized):
            fraction = [fraction]
        else:
            fraction = np.sort(fraction)[::-1]
        regularize_l2_norm(com, 0.0)
        results = []
        for fr in fraction:
            #print(min_growth)
            com.variables.community_objective.lb = fr * min_growth
            #com.variables.community_objective.ub = min_growth
            sol = solve(community, fluxes=True, pfba=False, atol=1e-4, rtol=1e-4)
            results.append((fr, sol))
        return pd.DataFrame.from_records(results, columns=["tradeoff", "solution"])


In [ ]:
def get_biomass_objectives_and_define_as_constraint(community):
    biomass_obj = []
    coefficients = dict()
    n_taxa = len(community.taxonomy)
    
    # get all biomass reactions
    for rec in community.reactions:
        if rec.id.startswith("Growth"):
            biomass_obj.append(rec)
            
    # coefficient scaled to abundance -> for equal abundance = 1/community_size
    for rxn in biomass_obj:
        coefficients[rxn.forward_variable] = 1/n_taxa
        coefficients[rxn.reverse_variable] = -1/n_taxa

    constraint = community.problem.Constraint(0, lb = 0, ub = None)
    community.add_cons_vars(constraint)
    community.solver.update()
    constraint.set_linear_coefficients(coefficients = coefficients)
    return constraint

In [ ]:
def add_pfba_objective_totalcom(community, minimal_growth, constraint, atol=1e-4, rtol=1e-4):
    """Add pFBA objective.

    Add objective to minimize the summed flux of all reactions to the
    current objective. This one will work with any objective (even non-linear
    ones).

    See Also
    --------
    pfba

    Parameters
    ----------
    community : micom.Community
        The community to add the objective to.
    """
    constraint.lb = (1-rtol) * minimal_growth - atol
    if community.solver.objective.name == "_pfba_objective":
        raise ValueError("model already has pfba objective")
    reaction_variables = (
        (rxn.forward_variable, rxn.reverse_variable) for rxn in community.reactions
    )
    variables = chain(*reaction_variables)
    community.objective = Zero
    community.objective_direction = "min"
    community.objective.set_linear_coefficients(dict.fromkeys(variables, 1.0))
    if community.modification is None:
        community.modification = "pFBA"
    else:
        community.modification += " and pFBA :)"
    community.solver.update()

In [ ]:
def run_fba(community, fractions, fluxes): #this is our custom pfba function 
    results = []
    opt_sol = community.optimize()
    for fr in fractions:
        with community:
            minimal_growth = opt_sol.growth_rate * fr
            biomass_constraint = get_biomass_objectives_and_define_as_constraint(community)
            add_pfba_objective_totalcom(community, minimal_growth, constraint = biomass_constraint, atol=1e-6, rtol=1e-6)

            community.solver.problem.parameters.advance.set(0)
            community.solver.problem.cleanup(1e-10)

            sol_pfba2 = community.optimize(fluxes=fluxes, raise_error = False)
            results.append((fr, sol_pfba2))

            print("---------")
            print(f"Tradeoff-value: {fr}")
            if sol_pfba2 != None:
                print(f"Community {community.solver.variables.community_objective.primal}")
                print(f"pFBA solution: {sol_pfba2.objective_value}")
            else:
                print(f"Solver status: {sol_pfba2}")
            print("---------")

    results_df = pd.DataFrame.from_records(results, columns=["tradeoff", "solution"])
    return results_df

### create communities

In [ ]:
HvSC1_df = taxonomy_df[(taxonomy_df["Syncom"] == "HvSC1") | (taxonomy_df["Syncom"] == "Both")]
HvSC1_df = HvSC1_df[["Strain ID","Family"]].rename(columns={'Strain ID': 'id', 'Family': 'family'}).copy()
HvSC1_df["id"] = HvSC1_df["id"].astype(str)
HvSC1_df["abundance"] = 1
HvSC1_df["file"] = HvSC1_df['id'].apply(lambda x: f"{model_dir1}/{x}_or_mb1_mdr_rdr_dp_mb2_lib_bz_fix.xml")

In [ ]:
HvSC1 = Community(HvSC1_df, id = "HvSC1", name = "HvSC1")

Output()

In [ ]:
HvSC2_df = taxonomy_df[(taxonomy_df["Syncom"] == "HvSC2") | (taxonomy_df["Syncom"] == "Both")]
HvSC2_df = HvSC2_df[["Strain ID","Family"]].rename(columns={'Strain ID': 'id', 'Family': 'family'}).copy()
HvSC2_df["id"] = HvSC2_df["id"].astype(str)
HvSC2_df["abundance"] = 1
HvSC2_df["file"] = HvSC2_df['id'].apply(lambda x: f"{model_dir2}/{x}_or_mb1_mdr_rdr_dp_mb2_lib_bz_fix.xml")

In [ ]:
HvSC2 = Community(HvSC2_df, id = "HvSC2", name = "HvSC2")

In [ ]:
af_path = os.path.join(media_dir, "af_diff_growth/combined_af7_c1i127.csv")
af_complete = pd.read_csv(af_path, names = ["reaction", "flux"])

In [ ]:
af_complete = create_medium(af_complete)

In [ ]:
HvSC1.medium = af_complete

In [ ]:
HvSC2.medium = af_complete

INFO:micom.community:I could not find the following exchanges in your model: EX_uacgam_m


In [ ]:
opt_sol1 = HvSC1.optimize()

In [ ]:
opt_sol2 = HvSC2.optimize()

### run different optimization methods

In [ ]:
fractions = [x / 10.0 for x in range(0, 11, 1)]

In [ ]:
complete_com = cooperative_tradeoff_custom(HvSC1, opt_sol1.objective_value, fractions) #test cooperative tradeoff for different values 

In [ ]:
complete_com2 = cooperative_tradeoff_custom(HvSC2, opt_sol2.objective_value, fractions) #test cooperative tradeoff for different values 

In [ ]:
complete_com_micom = cooperative_tradeoff(HvSC1, min_growth=0.0, fluxes=False,pfba=False, fraction=fractions, atol=1e-4, rtol=1e-4)

##### test for sugar blocks

In [ ]:
with HvSC1 as community:
    community.medium = af_complete
    e2m_rxn = [rxn for rxn in community.reactions if "EX_" in rxn.id and ("glc__D_e" in rxn.id or "cellb_e" in rxn.id)]
    fractions = [0.5]
    sol_opt_pre = community.optimize()
    print(f"optimize solution pre bloc: {sol_opt_pre.objective_value}")
    coop_sol_micom_pre = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False,  min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    coop_sol_custom_pre = cooperative_tradeoff_custom(community, fraction = fractions, min_growth =sol_opt_pre.objective_value)
    gr_pre = coop_sol_custom_pre.iloc[0]["solution"]
    print(f"Growth pre sugar block, custom: {gr_pre.growth_rate}")
    print(f"Growth pre sugar block, MICOM: {coop_sol_micom_pre.growth_rate}")


    for rxn in e2m_rxn:
        #print(f"PreBlock {rxn.id} bounds set to: {rxn.bounds}")
        community.reactions.get_by_id(rxn.id).bounds = (0, 0.0)
        #print(f"PostBlock {rxn.id} bounds set to: {rxn.bounds}")
    sol_opt = HvSC1.optimize()
    coop_sol_custom = cooperative_tradeoff_custom(community, fraction = fractions, min_growth = sol_opt.objective_value)
    coop_sol_micom = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False, min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    gr_post = coop_sol_custom.iloc[0]["solution"]
    print(f"Growth post sugar block, custom: {gr_post.growth_rate}")
    print(f"Growth post sugar block, MICOM: {coop_sol_micom.growth_rate}")

optimize solution pre bloc: 7.535269994625795
Growth pre sugar block, custom: 3.767635000673113
Growth pre sugar block, MICOM: 3.767635171176763
Growth post sugar block, custom: 3.5734187917554605
Growth post sugar block, MICOM: 3.5734194569336357


In [ ]:
with HvSC1 as community:
    community.medium = af_complete
    e2m_rxn = [rxn for rxn in community.reactions if "EX_" in rxn.id and ("glc__D_e" in rxn.id or "cellb_e" in rxn.id)]
    fractions = [0.5]
    sol_opt_pre = community.optimize()
    print(f"optimize solution pre bloc: {sol_opt_pre.objective_value}")
    coop_sol_micom_pre = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False,  min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    coop_sol_custom_pre = cooperative_tradeoff_custom(community, fraction = fractions, min_growth =sol_opt_pre.objective_value)
    gr_pre = coop_sol_custom_pre.iloc[0]["solution"]
    print(f"Growth pre sugar block, custom: {gr_pre.growth_rate}")
    print(f"Growth pre sugar block, MICOM: {coop_sol_micom_pre.growth_rate}")


    for rxn in e2m_rxn:
        #print(f"PreBlock {rxn.id} bounds set to: {rxn.bounds}")
        community.reactions.get_by_id(rxn.id).bounds = (0, 10.0)
        #print(f"PostBlock {rxn.id} bounds set to: {rxn.bounds}")
    sol_opt = HvSC1.optimize()
    coop_sol_custom = cooperative_tradeoff_custom(community, fraction = fractions, min_growth = sol_opt.objective_value)
    coop_sol_micom = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False, min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    gr_post = coop_sol_custom.iloc[0]["solution"]
    print(f"Growth post sugar block, custom: {gr_post.growth_rate}")
    print(f"Growth post sugar block, MICOM: {coop_sol_micom.growth_rate}")

optimize solution pre bloc: 7.535269994625795
Growth pre sugar block, custom: 3.767635000673113
Growth pre sugar block, MICOM: 3.767635171176763
Growth post sugar block, custom: 3.5734187718568395
Growth post sugar block, MICOM: 3.573418769979675


In [ ]:
with HvSC1 as community:
    community.medium = af_complete
    e2m_rxn = [rxn for rxn in community.reactions if "EX_" in rxn.id and ("glc__D_e" in rxn.id or "cellb_e" in rxn.id)]
    fractions = [0.5]
    sol_opt_pre = community.optimize()
    print(f"optimize solution pre bloc: {sol_opt_pre.objective_value}")
    coop_sol_micom_pre = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False,  min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    coop_sol_custom_pre = cooperative_tradeoff_custom(community, fraction = fractions, min_growth =sol_opt_pre.objective_value)
    gr_pre = coop_sol_custom_pre.iloc[0]["solution"]
    print(f"Growth pre sugar block, custom: {gr_pre.growth_rate}")
    print(f"Growth pre sugar block, MICOM: {coop_sol_micom_pre.growth_rate}")


    for rxn in e2m_rxn:
        #print(f"PreBlock {rxn.id} bounds set to: {rxn.bounds}")
        community.reactions.get_by_id(rxn.id).bounds = (-10.0, 0.0)
        #print(f"PostBlock {rxn.id} bounds set to: {rxn.bounds}")
    sol_opt = HvSC1.optimize()
    coop_sol_custom = cooperative_tradeoff_custom(community, fraction = fractions, min_growth = sol_opt.objective_value)
    coop_sol_micom = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False, min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    gr_post = coop_sol_custom.iloc[0]["solution"]
    print(f"Growth post sugar block, custom: {gr_post.growth_rate}")
    print(f"Growth post sugar block, MICOM: {coop_sol_micom.growth_rate}")

optimize solution pre bloc: 7.535269994625795
Growth pre sugar block, custom: 3.767635000673113
Growth pre sugar block, MICOM: 3.767635171176763
Growth post sugar block, custom: 3.726160168487522
Growth post sugar block, MICOM: 3.7261647140364933
